In [1]:
# ============================================== CONFIG & USER DEFINED FUNCTIONS ======================================================= #

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid

# =========================================================
# CONFIG
# =========================================================
def get_config(environment="dev"):
    config = {
        "dev": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        },
        "preprod": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        },
        "prod": {
            "raw_base_path": "Files/raw",
            "bronze_schema": "dbo",
            "silver_schema": "dbo",
            "gold_schema": "dbo"
        }
    }
    return config[environment]

CONTROL_LOG_TABLE = "ETL.pl_control_config_log"
BAD_RECORDS_TABLE = "ETL.pl_bad_records_log"
WATERMARK_TABLE = "ETL.pl_watermark_log"

def generate_run_id():
    return str(uuid.uuid4())

# =========================================================
# INSERT ONLY ONCE AT STEP START
# =========================================================
def log_step_start(
    run_id,
    pipeline_name,
    pipeline_step_name,
    source_file_path,
    target_table_name,
    pipeline_start_time
):
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("pipeline_step_name", StringType(), True),
        StructField("source_file_path", StringType(), True),
        StructField("target_table_name", StringType(), True),
        StructField("step_status", StringType(), True),
        StructField("pipeline_status", StringType(), True),
        StructField("record_count", IntegerType(), True),
        StructField("error_message", StringType(), True),
        StructField("step_start_time", TimestampType(), True),
        StructField("step_end_time", TimestampType(), True),
        StructField("pipeline_start_time", TimestampType(), True),
        StructField("pipeline_end_time", TimestampType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ])

    now = datetime.now()

    row = [(
        run_id,
        pipeline_name,
        pipeline_step_name,
        source_file_path,
        target_table_name,
        "STARTED",
        "STARTED",
        0,
        None,
        now,
        None,
        pipeline_start_time,
        None,
        now,
        now
    )]

    df = spark.createDataFrame(row, schema)
    df.write.format("delta").mode("append").saveAsTable(CONTROL_LOG_TABLE)

# =========================================================
# UPDATE SAME ROW AT STEP END
# =========================================================
def log_step_end(
    run_id,
    pipeline_name,
    pipeline_step_name,
    step_status,
    pipeline_status,
    record_count,
    error_message=None,
    pipeline_end_time=None
):
    error_message = (error_message or "").replace("'", " ")

    if pipeline_end_time is None:
        pipeline_endt = "NULL"
    else:
        pipeline_endt = f"TIMESTAMP('{pipeline_end_time}')"

    spark.sql(f"""
        UPDATE {CONTROL_LOG_TABLE}
        SET step_status = '{step_status}',
            pipeline_status = '{pipeline_status}',
            record_count = {record_count},
            error_message = '{error_message}',
            step_end_time = current_timestamp(),
            pipeline_end_time = {pipeline_endt},
            updated_at = current_timestamp()
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
          AND pipeline_step_name = '{pipeline_step_name}'
    """)

# =========================================================
# OPTIONAL: UPDATE FINAL PIPELINE STATUS FOR ALL ROWS
# =========================================================
def finalize_pipeline_run(run_id, pipeline_name, final_status):
    spark.sql(f"""
        UPDATE {CONTROL_LOG_TABLE}
        SET pipeline_status = '{final_status}',
            pipeline_end_time = current_timestamp(),
            updated_at = current_timestamp()
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
    """)

# =========================================================
# RESTART / SKIP LOGIC
# =========================================================
def should_run_step(run_id, pipeline_name, pipeline_step_name, rerun_mode="FULL"):
    if rerun_mode.upper() == "FULL":
        return True

    df = spark.sql(f"""
        SELECT step_status
        FROM {CONTROL_LOG_TABLE}
        WHERE run_id = '{run_id}'
          AND pipeline_name = '{pipeline_name}'
          AND pipeline_step_name = '{pipeline_step_name}'
        ORDER BY updated_at DESC
        LIMIT 1
    """)

    rows = df.collect()

    if len(rows) == 0:
        return True

    latest_status = rows[0]["step_status"]

    if rerun_mode.upper() == "FAILED_ONLY" and latest_status == "SUCCESS":
        return False

    return True

# =========================================================
# DATE PARSER
# =========================================================
def parse_multi_format_timestamp(col_name):
    return F.coalesce(
        F.to_timestamp(F.col(col_name), "yyyy-MM-dd"),
        F.to_timestamp(F.col(col_name), "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(F.col(col_name), "dd/MM/yyyy"),
        F.to_timestamp(F.col(col_name), "MM/dd/yyyy"),
        F.to_timestamp(F.col(col_name), "yyyy/MM/dd"),
        F.to_timestamp(F.col(col_name), "dd-MM-yyyy"),
        F.to_timestamp(F.col(col_name))
    )

# =========================================================
# DELTA MERGE
# =========================================================
def merge_upsert(df, target_table_name, merge_condition):
    if spark.catalog.tableExists(target_table_name):
        delta_table = DeltaTable.forName(spark, target_table_name)
        (
            delta_table.alias("t")
            .merge(df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(target_table_name)

# ============================================== CONFIG & USER DEFINED FUNCTIONS ======================================================= #

StatementMeta(, 504cbaeb-829c-410d-8f1a-18a37b4f2d3b, 9, Finished, Available, Finished, False)

In [2]:
# =========================================================
# PARAMETERS
# =========================================================
pipeline_start_time = datetime.now()
pipeline_name = "pl_retail_analytics_medallion"

environment = globals().get("environment", "dev")
run_id = globals().get("run_id", generate_run_id())
rerun_mode = globals().get("rerun_mode", "FULL")   # FULL / FAILED_ONLY

config = get_config(environment)
raw_base_path = config["raw_base_path"]

print(f"environment = {environment}")
print(f"run_id = {run_id}")
print(f"rerun_mode = {rerun_mode}")

StatementMeta(, 504cbaeb-829c-410d-8f1a-18a37b4f2d3b, 10, Finished, Available, Finished, False)

environment = dev
run_id = 45e6d27f-77cb-49a0-b1ae-7eecf73012bb
rerun_mode = FULL


In [3]:
# =========================================================
# SOURCE METADATA
# Bronze must ingest exactly as received
# =========================================================
source_metadata = [
    {
        "step_name": "bronze_customers",
        "file_name": "customers.csv",
        "format": "csv",
        "multiline": False,
        "target_table": "dbo.bronze_customers"
    },
    {
        "step_name": "bronze_products",
        "file_name": "products.csv",
        "format": "csv",
        "multiline": False,
        "target_table": "dbo.bronze_products"
    },
    {
        "step_name": "bronze_orders",
        "file_name": "orders.csv",
        "format": "csv",
        "multiline": False,
        "target_table": "dbo.bronze_orders"
    },
    {
        "step_name": "bronze_order_items",
        "file_name": "order_items.json",
        "format": "json",
        "multiline": True,
        "target_table": "dbo.bronze_order_items"
    },
    {
        "step_name": "bronze_returns",
        "file_name": "returns.csv",
        "format": "csv",
        "multiline": False,
        "target_table": "dbo.bronze_returns"
    }
]


StatementMeta(, 504cbaeb-829c-410d-8f1a-18a37b4f2d3b, 11, Finished, Available, Finished, False)

In [4]:
# =========================================================
# BRONZE LOAD LOOP
# =========================================================
for item in source_metadata:
    step_name = item["step_name"]
    file_name = item["file_name"]
    file_format = item["format"]
    multiline = item["multiline"]
    target_table = item["target_table"]

    source_file_path = f"{raw_base_path}/{file_name}"
    step_start_time = datetime.now()

    # -----------------------------------------------------
    # Restart logic
    # -----------------------------------------------------
    if not should_run_step(run_id, pipeline_name, step_name, rerun_mode):
        print(f"{step_name} skipped")
        continue

    try:
        # -------------------------------------------------
        # Log STARTED
        # -------------------------------------------------
        log_step_start(
            run_id=run_id,
            pipeline_name=pipeline_name,
            pipeline_step_name=step_name,
            source_file_path=source_file_path,
            target_table_name=target_table,
            pipeline_start_time = pipeline_start_time
        )

        # -------------------------------------------------
        # Read source exactly as received
        # -------------------------------------------------
        if file_format == "csv":
            df = (
                spark.read
                     .option("header", True)
                     .csv(source_file_path)
            )
        elif file_format == "json":

            df = (
                spark.read
                     .option("multiLine", multiline)
                     .json(source_file_path)
            )
        else:
            raise Exception(f"Unsupported file format: {file_format}")

        # -------------------------------------------------
        # Add required audit columns only
        # -------------------------------------------------
        bronze_df = (
            df.withColumn("ingestion_ts", F.current_timestamp())
              .withColumn("source_file_name", F.input_file_name())
              .withColumn("pipeline_run_id", F.lit(run_id))
        )

        record_count = bronze_df.count()

        # -------------------------------------------------
        # Write Bronze Delta table
        # overwrite is fine for assignment/demo reruns
        # -------------------------------------------------
        bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

        # -------------------------------------------------
        # Log SUCCESS
        # -------------------------------------------------
        log_step_end(
            run_id=run_id,
            pipeline_name=pipeline_name,
            pipeline_step_name=step_name,
            step_status="SUCCESS",
            pipeline_status="IN_PROGRESS",
            record_count=record_count
        )

        print(f"{step_name} loaded successfully. record_count = {record_count}")

    except Exception as e:
        # -------------------------------------------------
        # Technical failure only - log in control table
        # -------------------------------------------------
        log_step_end(
            run_id=run_id,
            pipeline_name=pipeline_name,
            pipeline_step_name=step_name,
            step_status="FAILED",
            pipeline_status="FAILED",
            record_count=0,
            error_message=str(e),
            pipeline_end_time = datetime.now()
        )

        raise

print("Bronze ingestion completed successfully")
print(f"pipeline_run_id = {run_id}")

StatementMeta(, 504cbaeb-829c-410d-8f1a-18a37b4f2d3b, 12, Finished, Available, Finished, False)

bronze_customers loaded successfully. record_count = 6
bronze_products loaded successfully. record_count = 4
bronze_orders loaded successfully. record_count = 5
bronze_order_items loaded successfully. record_count = 6
bronze_returns loaded successfully. record_count = 2
Bronze ingestion completed successfully
pipeline_run_id = 45e6d27f-77cb-49a0-b1ae-7eecf73012bb
